# IoT Water Quality Safety Classifier (Project ML_17)
### A Machine Learning Pipeline for Water Potability Prediction
**Teacher's Guide:** Welcome to your college machine learning lab project! In this project, we build a complete, leak-free machine learning workflow to classify whether water is safe for human consumption (**Potable**, class `1`) or unsafe (**Non-Potable**, class `0`).

#### Project Roadmap:
1. **Setup & Imports**: Load verified scientific packages and configure dynamic path resolution.
2. **Dataset Inspection**: Explore shape, data types, missing values, duplicates, and target classes.
3. **Exploratory Data Analysis (EDA)**: Visualize class balance, feature distributions, missingness, and correlation.
4. **Leak-Free Data Preparation**: Stratified train/test split, keeping test data completely untouched, and using pipelines for preprocessing.
5. **Model Training & Comparison**: Baseline DummyClassifier, Tuned k-Nearest Neighbors (k-NN) using Stratified 5-Fold GridSearchCV, and Gaussian Naive Bayes.
6. **Final Evaluation**: Test-set metrics (Accuracy, Precision, Recall, F1), confusion matrix, and in-depth false positive/false negative analysis.
7. **Model Serialization**: Save fitted pipeline with metadata via joblib, and test inference on a new sample.

## 1. Setup & Environment
We import pandas, NumPy, Matplotlib, Seaborn, scikit-learn, and joblib.
We also define a helper function to dynamically locate the dataset whether the notebook is executed from `notebooks/` or the project root.

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='muted')
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    plt.style.use('default')

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import joblib

plt.rcParams['figure.figsize'] = (10, 6)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")
print(f"Scikit-Learn:   {sklearn.__version__}")
print(f"Joblib Version: {joblib.__version__}")
print(f"Seaborn Installed: {HAS_SEABORN}")

## 2. Dataset Inspection & Verification

### Dataset Provenance & Metadata:
- **Title:** Water Quality / Water Potability Dataset
- **Publisher:** Aditya Kadiwal (Kaggle)
- **License:** CC0: Public Domain (open access)
- **Target Definition:** `Potability` (`1` = Safe for drinking, `0` = Not safe for drinking)

### Features & Standard Units:
1. `ph`: Acidity/Alkalinity level (dimensionless, 0–14 scale; WHO standard: 6.5–8.5).
2. `Hardness`: Capacity to precipitate soap, primarily calcium and magnesium ($mg/L$).
3. `Solids`: Total Dissolved Solids / TDS ($ppm$ or $mg/L$).
4. `Chloramines`: Disinfectant concentration from chlorine + ammonia treatment ($ppm$).
5. `Sulfate`: Dissolved sulfate minerals ($mg/L$).
6. `Conductivity`: Electrical conductivity from dissolved ions ($\mu S/cm$).
7. `Organic_carbon`: Total organic carbon content ($ppm$).
8. `Trihalomethanes`: Disinfection byproducts formed when chlorine reacts with organic matter ($\mu g/L$).
9. `Turbidity`: Measure of optical clarity / suspended cloudiness ($NTU$).

In [ ]:
def find_dataset(filename="water_potability.csv"):
    candidates = [
        Path("data") / filename,
        Path("../data") / filename,
        Path("water-quality-classifier/data") / filename,
        Path.cwd() / "data" / filename,
        Path.cwd().parent / "data" / filename
    ]
    for p in candidates:
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(f"Could not locate {filename} in data/ directory.")

dataset_path = find_dataset("water_potability.csv")
print(f"Loading verified CSV from: {dataset_path}")

df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### Verifying Data Types, Missing Values, and Duplicates
Let us check:
1. Data types (all numerical features).
2. Missing values across features.
3. Existence of duplicated rows.
4. Target class validity (must only contain 0 or 1).

In [ ]:
print("--- Column Data Types ---")
print(df.dtypes)

print("\n--- Missing Values Summary ---")
missing_summary = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing_summary[missing_summary['Missing Values'] > 0])

print(f"\nExact Duplicate Rows: {df.duplicated().sum()}")

print("\n--- Target Class Counts (Potability) ---")
print(df['Potability'].value_counts(dropna=False))

print("\n--- Target Class Proportions ---")
print(df['Potability'].value_counts(normalize=True).round(4) * 100)

# Validate target labels and drop any invalid/missing targets if found
valid_rows = df['Potability'].isin([0, 1])
if not valid_rows.all():
    print(f"Warning: Found {(~valid_rows).sum()} rows with invalid target labels. Dropping.")
    df = df[valid_rows].copy()
else:
    print("Verification Passed: 100% of target labels are valid binary values (0 or 1).")

### Summary Statistics
Let us compute the statistical properties (mean, std, min, median, max) of our features.

In [ ]:
df.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]

## 3. Exploratory Data Analysis (EDA)

Exploratory Data Analysis helps us understand distributions, skewness, scale differences, and target balance.

### 3.1 Target Class Balance
Understanding class balance tells us whether standard accuracy is a fair metric or if class imbalance requires special care.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Subplot 1: Bar Chart
counts = df['Potability'].value_counts()
axes[0].bar(['Non-Potable (0)', 'Potable (1)'], counts.values, color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Target Class Counts', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 25, str(v), ha='center', fontweight='bold')

# Subplot 2: Donut Chart
axes[1].pie(
    counts,
    labels=['Non-Potable (0)', 'Potable (1)'],
    autopct='%1.1f%%',
    colors=['#e74c3c', '#2ecc71'],
    startangle=140,
    wedgeprops=dict(width=0.6, edgecolor='white')
)
axes[1].set_title('Target Class Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

**Observation:**
- Non-potable water represents **60.99%** (1,998 samples).
- Potable water represents **39.01%** (1,278 samples).
Because the majority class represents ~61%, a naive baseline predicting only "Non-Potable" achieves ~61% accuracy. Any useful machine learning model must perform better than this baseline and capture both classes meaningfully.

### 3.2 Visualizing Missing Values
Three features contain missing values:
- `ph`: 491 missing (~15.0%)
- `Sulfate`: 781 missing (~23.8%)
- `Trihalomethanes`: 162 missing (~4.9%)

In [ ]:
missing_features = df.isnull().sum()[df.isnull().sum() > 0].sort_values(ascending=False)
plt.figure(figsize=(9, 4))
plt.barh(missing_features.index, missing_features.values, color='#3498db')
plt.title('Missing Value Counts by Feature', fontsize=13, fontweight='bold')
plt.xlabel('Count of Missing Values')
for idx, val in enumerate(missing_features.values):
    plt.text(val + 10, idx, f"{val} ({val/len(df)*100:.1f}%)", va='center', fontweight='bold')
plt.xlim(0, 950)
plt.gca().invert_yaxis()
plt.show()

### 3.3 Feature Distributions
Let us inspect the distribution of all 9 numerical features.

In [ ]:
features = [c for c in df.columns if c != 'Potability']
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(features):
    vals = df[col].dropna()
    axes[i].hist(vals, bins=30, color='#2980b9', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribution of {col}', fontweight='bold')
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.show()

**Observation on Scale:**
Notice that `Solids` ranges up to 61,000 $ppm$, while `ph` is between 0 and 14. If we fed unscaled data into k-NN, distance would be dominated by `Solids`, completely ignoring `ph`, `turbidity`, and other critical measurements. **Feature scaling (`StandardScaler`) is mathematically required for distance-based k-NN.**

### 3.4 Correlation Matrix Heatmap
Let us analyze linear relationships between features and the target.

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()

if HAS_SEABORN:
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask, square=True, linewidths=0.5)
else:
    cax = plt.matshow(corr, cmap='coolwarm', fignum=1)
    plt.colorbar(cax)
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    for i in range(len(corr.columns)):
        for j in range(len(corr.columns)):
            plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center', color='black', fontsize=9)

plt.title('Correlation Matrix of Water Quality Parameters', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

**Key Finding:**
The correlation of all features with `Potability` is virtually zero (between -0.03 and +0.02). This proves that water safety is governed by complex, multi-factor non-linear boundaries rather than simple individual linear relationships.

## 4. Leak-Free Data Preparation

### The Principle of Preventing Data Leakage
Data leakage occurs when information outside the training fold is used during preprocessing.
To adhere strictly to best practices:
1. Split the dataset **first** using a stratified split (80% train, 20% test, random_state=42).
2. Lock away the test set completely until final evaluation.
3. Embed preprocessing steps (`SimpleImputer` and `StandardScaler`) inside a `scikit-learn Pipeline`.
4. The pipeline fits its imputer medians and scaling parameters **only on the training fold** during cross-validation.

In [ ]:
# 4.1 Separate features X and target y
X = df.drop(columns=['Potability']).copy()
y = df['Potability'].copy()

# 4.2 Verify no duplicate rows exist
print(f"Identical duplicate rows in X: {X.duplicated().sum()}")

# 4.3 Stratified train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} rows, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} rows, {X_test.shape[1]} features")

print("\n--- Target Proportions in Train vs Test ---")
print("Train Set Potable %:", round(y_train.mean() * 100, 2))
print("Test Set Potable %: ", round(y_test.mean() * 100, 2))

## 5. Model Training, Hyperparameter Tuning & Cross-Validation

We will train and compare:
1. **DummyClassifier Baseline**: Predicts the most frequent class.
2. **k-Nearest Neighbors (k-NN) Pipeline**: Preprocessing + k-NN tuned via `GridSearchCV` with 5-fold Stratified Cross-Validation.
3. **Gaussian Naive Bayes (GaussianNB)**: Analytical probabilistic comparison.
4. **Model Selection**: Selected strictly using training-data cross-validation accuracy.

In [ ]:
# Setup 5-Fold Stratified Cross-Validation
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1. Dummy Baseline
dummy_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('dummy', DummyClassifier(strategy='most_frequent'))
])
dummy_cv = cross_val_score(dummy_pipe, X_train, y_train, cv=cv_stratified, scoring='accuracy')

print("=== 1. Baseline: Dummy Classifier ===")
print(f"5-Fold CV Accuracy: {dummy_cv.mean():.4f} (+/- {dummy_cv.std():.4f})")

### 5.2 k-NN Pipeline & GridSearchCV
We tune:
- `n_neighbors`: [3, 5, 7, 9, 11, 15, 21, 25, 31]
- `weights`: `'uniform'` (all neighbors vote equally) vs `'distance'` (closer neighbors vote with higher weight)
- `metric`: `'euclidean'` ($L_2$ distance) vs `'manhattan'` ($L_1$ distance)

In [ ]:
knn_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid_knn = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 21, 25, 31],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

print("Running Stratified 5-Fold GridSearchCV on Training Data...")
grid_knn = GridSearchCV(
    estimator=knn_pipe,
    param_grid=param_grid_knn,
    cv=cv_stratified,
    scoring='accuracy',
    n_jobs=-1
)
grid_knn.fit(X_train, y_train)

print(f"Best k-NN Parameters: {grid_knn.best_params_}")
print(f"Best k-NN 5-Fold CV Accuracy: {grid_knn.best_score_:.4f}")

### 5.3 Gaussian Naive Bayes Pipeline
Gaussian Naive Bayes applies Bayes\' theorem under the conditional independence assumption.

In [ ]:
gnb_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('gnb', GaussianNB())
])

gnb_cv = cross_val_score(gnb_pipe, X_train, y_train, cv=cv_stratified, scoring='accuracy')

print("=== 2. Comparison Model: Gaussian Naive Bayes ===")
print(f"5-Fold CV Accuracy: {gnb_cv.mean():.4f} (+/- {gnb_cv.std():.4f})")

### 5.4 Cross-Validation Comparison & Model Selection
Let us summarize training cross-validation metrics:

In [ ]:
comparison_df = pd.DataFrame({
    'Model': [
        'Dummy Classifier (Baseline)',
        'Gaussian Naive Bayes',
        f"k-NN (Tuned: {grid_knn.best_params_['knn__n_neighbors']} neighbors, {grid_knn.best_params_['knn__weights']})"
    ],
    '5-Fold CV Accuracy': [dummy_cv.mean(), gnb_cv.mean(), grid_knn.best_score_],
    'CV Std Dev': [dummy_cv.std(), gnb_cv.std(), grid_knn.cv_results_['std_test_score'][grid_knn.best_index_]]
})

print("Cross-Validation Comparison Table (Training Folds Only):")
print(comparison_df.to_string(index=False))

# Select final model based strictly on CV performance
selected_pipeline = grid_knn.best_estimator_
print(f"\nFinal Selected Model: Tuned k-NN Pipeline (CV Accuracy: {grid_knn.best_score_:.4f})")

## 6. Final Evaluation on Held-Out Test Set

We now evaluate the selected k-NN pipeline on the untouched test set (`X_test`, `y_test`).

In [ ]:
# Evaluate selected pipeline on held-out test data
y_pred = selected_pipeline.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
test_prec = precision_score(y_test, y_pred, zero_division=0)
test_rec = recall_score(y_test, y_pred, zero_division=0)
test_f1 = f1_score(y_test, y_pred, zero_division=0)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("=" * 55)
print("             FINAL HELD-OUT TEST EVALUATION")
print("=" * 55)
print(f"Test Accuracy:          {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Precision (Class 1):    {test_prec:.4f} ({test_prec*100:.2f}%)")
print(f"Recall (Class 1):       {test_rec:.4f} ({test_rec*100:.2f}%)")
print(f"F1-Score (Class 1):     {test_f1:.4f}")
print("-" * 55)
print(f"Total Test Samples:     {len(y_test)}")
print(f"Correct Predictions:    {tn + tp} ({((tn + tp)/len(y_test))*100:.2f}%)")
print(f"Incorrect Predictions:  {fp + fn} ({((fp + fn)/len(y_test))*100:.2f}%)")
print("=" * 55)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Non-Potable (0)', 'Potable (1)'], digits=4))

### Confusion Matrix Visualization & Error Analysis
Let us plot the test set confusion matrix and analyze the practical risks of False Positives versus False Negatives.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Potable (0)', 'Potable (1)'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title(f'Held-Out Test Confusion Matrix (Accuracy: {test_acc:.2%})', fontsize=13, fontweight='bold')
plt.grid(False)
plt.show()

print(f"True Negatives (TN):  {tn} (Correctly classified as Non-Potable)")
print(f"False Positives (FP): {fp} (Non-Potable incorrectly classified as Potable -> SAFETY HAZARD)")
print(f"False Negatives (FN): {fn} (Potable incorrectly classified as Non-Potable)")
print(f"True Positives (TP):  {tp} (Correctly classified as Potable)")

### Critical Discussion: Real-World Safety Implications
In drinking water safety classification:
1. **The False Positive Hazard (FP = 32 samples):**
   A False Positive means non-potable (contaminated) water was predicted to be safe. If a community consumed water based solely on this prediction, people would drink contaminated water, risking gastrointestinal illnesses or heavy-metal poisoning. **In public health ML, False Positives are the highest-risk failure mode.**
2. **The False Negative Inconvenience (FN = 204 samples):**
   A False Negative means safe water was predicted to be non-potable. This causes water to be unnecessarily discarded or subjected to extra boiling or filtration, causing economic cost but no human health hazard.
3. **Honest Reporting of Modest Accuracy (~64%):**
   We do not claim 90%+ accuracy because tabular physical-chemical measurements without biological assays (such as microbial pathogen tests) have inherent overlapping distributions. Reporting real results demonstrates scientific integrity.

## 7. Model Persistence & Inference Verification

We serialize the fitted pipeline along with all feature metadata, label definitions, and library versions into `models/water_quality_pipeline.joblib`.

In [ ]:
# Determine models directory
models_dir = Path("models")
if not models_dir.exists():
    models_dir = Path("../models")
if not models_dir.exists():
    models_dir = Path("water-quality-classifier/models")
models_dir.mkdir(parents=True, exist_ok=True)

model_filepath = models_dir / "water_quality_pipeline.joblib"

# Bundle metadata with pipeline
model_artifact = {
    'pipeline': selected_pipeline,
    'feature_names': list(X.columns),
    'feature_order': list(X.columns),
    'target_column': 'Potability',
    'class_labels': {
        0: 'Non-Potable (Unsafe to drink)',
        1: 'Potable (Safe to drink)'
    },
    'hyperparameters': grid_knn.best_params_,
    'training_cv_accuracy': float(grid_knn.best_score_),
    'test_accuracy': float(test_acc),
    'scikit_learn_version': sklearn.__version__
}

joblib.dump(model_artifact, model_filepath)
print(f"Pipeline successfully saved to: {model_filepath.resolve()}")

### Verification: Loading the Model and Predicting on a New Sample
We test reloading the pipeline from disk and generating predictions for an example sample.

In [ ]:
# Reload artifact
loaded_artifact = joblib.load(model_filepath)
loaded_pipe = loaded_artifact['pipeline'] if isinstance(loaded_artifact, dict) else loaded_artifact

# Define a simulated water sample
sample_record = pd.DataFrame([{
    'ph': 7.15,
    'Hardness': 185.0,
    'Solids': 19500.0,
    'Chloramines': 6.8,
    'Sulfate': 325.0,
    'Conductivity': 415.0,
    'Organic_carbon': 13.5,
    'Trihalomethanes': 62.0,
    'Turbidity': 3.7
}])

prediction = loaded_pipe.predict(sample_record)[0]
probabilities = loaded_pipe.predict_proba(sample_record)[0]

print("--- Standalone Prediction Test ---")
print(f"Input Sample:\n{sample_record.iloc[0].to_dict()}")
print(f"\nModel Prediction: {prediction} -> {loaded_artifact['class_labels'][prediction]}")
print(f"Estimated Probabilities -> Non-Potable: {probabilities[0]:.2%}, Potable: {probabilities[1]:.2%}")